In [ ]:
import os
import torch
import random
import numpy as np
import pandas as pd
import robotic as ry
from collections import Counter
from scipy.spatial.transform import Rotation as R
import open3d as o3d

import WayTu_RAI.model_utils as mutils
import WayTu_RAI.graph_utils as gutils

from WayTu_RAI.GenerateEnvironment import GenerateEnvironment, PlacementError
from WayTu_RAI.generator_and_selector import WayTuUnifiedModel 


ry.params_add({'physx/motorKp': 10000., 
               'physx/motorK  d': 1000., 
               'physx/angularDamping': 10., 
               'physx/defaultFriction': 1000.})

ry.params_add({'botsim/engine': 'physx'}) 
ry.params_add({'physx/multibody': True}) 
ry.params_print()

print(ry.__version__)




def draw_pointcloud(env_pc):
    xyz = np.ascontiguousarray(env_pc)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(xyz)

    o3d.visualization.draw_geometries([pcd])


def unified_model_selection_test(parameters):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path = os.path.join("models", parameters["model-name"])
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    model = WayTuUnifiedModel(parameters, device).to(device)
    # model.load_state_dict(torch.load(model_path, map_location=device))
    checkpoint = torch.load(model_path, map_location=device)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint

    model.load_state_dict(state_dict)
    model.eval()

    label_list = parameters["label-list-all"]
    env_label_idx = [i for i, name in enumerate(label_list) if "platform" in name]
    tool_label_idx = [i for i, name in enumerate(label_list) if "platform" not in name]

    log_data = []
    trial = 0
    success_count = 0
    grasp_success = 0

    save_path = parameters['dataset-save-path']
    if not os.path.exists(os.path.join("./", save_path)):
        os.mkdir(os.path.join("./", save_path))
    current_sample_count = len(os.listdir(save_path))


    while trial < parameters["num-trials"]:
        try:
            print(f"\n=== Test Trial {trial + 1} ===")
            environment = GenerateEnvironment(parameters)
            environment.generate_environment()


            env_pcl = environment.point_clouds_labels
            pcl = mutils.uniform_object_point_sampling(env_pcl, parameters["num-obj-points"])

            xyz = pcl[:, :3]
            labels = pcl[:, 3].astype(int)

            # draw_pointcloud(xyz)

            tool_mask = np.isin(labels, tool_label_idx)
            env_mask = np.isin(labels, env_label_idx)

            tool_scores = {}
            tool_waypoints = {}

            env_pc = xyz[env_mask]
            x_values = env_pc[:, 0]
            mean_x = np.mean(x_values)

            # draw_pointcloud(env_pc)
            
            hammering_flag = False
            if mean_x < 0 and parameters["task"] in ["hammering"]:
                env_pc[:, 0] = -env_pc[:, 0]
                hammering_flag = True

            env_center = env_pc.mean(axis=0)
            env_pc_norm = env_pc - env_center
            env_scale = np.linalg.norm(env_pc_norm, axis=1).max()
            env_pc_norm = env_pc_norm / env_scale

            env_labels_in_sample = labels[env_mask]
            platform_label = Counter(env_labels_in_sample).most_common(1)[0][0]
            platform_class_idx = env_label_idx.index(platform_label)
            env_onehot = np.zeros(len(env_label_idx), dtype=np.float32)
            env_onehot[platform_class_idx] = 1.0
            env_onehot_tensor = torch.tensor(env_onehot, dtype=torch.float32).unsqueeze(0).to(device)

            for tool_idx in tool_label_idx:
                tool_points = xyz[labels == tool_idx]
                if len(tool_points) < 10:
                    continue

                tool_center = tool_points.mean(axis=0)
                tool_points_norm = tool_points - tool_center
                tool_scale = np.linalg.norm(tool_points_norm, axis=1).max()
                tool_points_norm = tool_points_norm / tool_scale
                tool_tensor = torch.tensor(tool_points_norm, dtype=torch.float32).unsqueeze(0).to(device)

                env_tensor = torch.tensor(env_pc_norm, dtype=torch.float32).unsqueeze(0).to(device)
                # target_center = mutils.get_target_center(torch.tensor(env_pc_norm), platform_class_idx)
                target_center = gutils.get_target_center(torch.tensor(env_pc), platform_class_idx)
                print(f"**DEBUG** target_center old: {target_center}")
                print(f"**DEBUG** env_center old: {env_center}")
                lifting_side_fix = False
                if target_center[0] < 0 and parameters["task"] == "lifting":
                    lifting_side_fix = True
                    print(f"**DEBUG** I am in the size fix block")
                if lifting_side_fix:
                    target_center[0] = - target_center[0]
                    env_center[0] = - env_center[0]

                print(f"**DEBUG** target_center new: {target_center}")
                print(f"**DEBUG** env_center new: {env_center}")

                
                params = {
                    "tool_centers": torch.tensor(tool_center, dtype=torch.float32).unsqueeze(0).to(device),
                    "tool_scales": torch.tensor(tool_scale, dtype=torch.float32).unsqueeze(0).to(device),
                    "env_centers": torch.tensor(env_center, dtype=torch.float32).unsqueeze(0).to(device),
                    "env_scales": torch.tensor(env_scale, dtype=torch.float32).unsqueeze(0).to(device),
                    "target_center": torch.tensor(target_center, dtype=torch.float32).unsqueeze(0).to(device),
                }
                with torch.no_grad():
                    pos, quat, score, yaw = model(tool_tensor, env_tensor, env_onehot_tensor, params)
                print(f"target_center: {target_center}")
                print(f"real target_center: {environment.C.getFrame(parameters['task'] + '-obj').getPosition()}")
               

                # -- Denormalize the positions ---- 
                tool_center = params["tool_centers"][0]  # [3]
                tool_scale = params["tool_scales"][0]    # [3]
                env_center = params["env_centers"][0]    # [3]
                env_scale = params["env_scales"][0]      # [3]

                # Separate the tool and environment waypoints
                tool_pos = pos[0, 0] * tool_scale + tool_center
                env_pos1 = pos[0, 1] * env_scale + env_center
                env_pos2 = pos[0, 2] * env_scale + env_center


                pos = torch.stack([tool_pos, env_pos1, env_pos2], dim=0)

                score = score.item()
                tool_name = label_list[tool_idx]

                tool_scores[tool_name] = score
                tool_waypoints[tool_name] = {
                    "pos": pos.squeeze().cpu().numpy(),
                    "qua": quat.squeeze().cpu().numpy(),
                    "yaw": yaw.squeeze().cpu().numpy()
                }

                print(f"Tool: {tool_name}, Score: {score:.4f}")

            if not tool_scores:
                print("[!] No tools found in scene.")
                continue

            best_tool = max(tool_scores, key=tool_scores.get)
            print(f"best_tool: {best_tool}")
            best_score = tool_scores[best_tool]
            best_wp = tool_waypoints[best_tool]

            print("best_tool: ", best_tool)
            print("best_wp: ", best_wp)
            print(f"best tool yaw: {mutils.yaw_to_quaternion_scalar_first(best_wp['yaw'])}")

            print(f"==> Selected Tool: {best_tool} with score {best_score:.4f}")
            # selected_part = mutils.find_grasping_part(environment.C, best_tool, np.concatenate([best_wp["pos"][0],best_wp["qua"][0]]))
            pos = np.array(best_wp["pos"]).reshape(-1)
            qua = np.array(best_wp["qua"]).reshape(-1)

            selected_part = mutils.find_grasping_part(
                environment.C,
                best_tool,
                best_wp["pos"][0]
            )

            # Debug: 
            print(pos)
            selected_tool_idx = label_list.index(best_tool)
            best_wp = mutils.post_process_waypoints_for_tasks(
                task=parameters["task"],
                C=environment.C,
                selected_tool=best_tool,
                selected_part=selected_part,
                best_wp=best_wp,
                tool_points=xyz[labels == selected_tool_idx],
                target_center=target_center,
                lifting_side_fix=lifting_side_fix,
                hammering_flag=hammering_flag,
            )
            # ---------------------------------------------------------------
            tool_waypoint = np.concatenate([pos, qua])
            selected_part = mutils.find_grasping_part(
                environment.C,
                best_tool,
                tool_waypoint
            )
            print(f"**DEBUG** Selected part: {selected_part}")
            other_tools = [item for item in environment.tool_objs if item != best_tool]
            mutils.reparent_tool(environment.C, best_tool, selected_part, other_tools)
            environment.add_model_waypoints(best_wp)

            environment.C.view()

            
            data_path = os.path.join(parameters['dataset-save-path'], "data_" + str(current_sample_count + trial) + "_initial.g")
            environment.save_environment_path(data_path)

            real_score = mutils.run_manipulation_for_task(
                task=parameters["task"],
                C=environment.C,
                selected_tool=best_tool,
                task_environment=environment.env,
                environment_controller=environment,
            )
            
            if real_score["score"] >= 0.5 and real_score["grasp_score"] >= 0.2 and real_score["task_score"] >= 0.2:
                success_count += 1
            if real_score["grasp_score"] >= 0.3:
                grasp_success += 1
            
            print(best_wp)
            log_data.append({
                "trial": trial + 1,
                "selected_tool": best_tool,
                "score": best_score,
                "real_score": real_score["score"],
                "grasp_score": real_score["grasp_score"],
                "task_score": real_score["task_score"]

            })
            # environment.save_environment(current_sample_count + trial)
            data_path = os.path.join(parameters['dataset-save-path'], "data_" + str(current_sample_count + trial) + "_final.g")
            environment.save_environment_path(data_path)
            trial += 1
            

        except PlacementError as e:
            print(f"[!] Skipping trial due to placement error: {e}")
            continue

    df = pd.DataFrame(log_data)
    df.to_csv("tool_selection_test_log.csv", index=False)
    print("\nTest complete. Results saved to tool_selection_test_log.csv")

    selected_tools = [row["selected_tool"] for row in log_data]
    tool_counts = Counter(selected_tools)
    print("\n=== Tool Selection Frequency ===")
    for tool, count in tool_counts.items():
        print(f"{tool}: {count} times")
     
    print(f"Grasp Sucess count: {grasp_success}")
    print(f"Success count: {success_count}")

    return df


In [ ]:
parameters = { 
    "mode" : "test", 
    "num-tools" : 1,
    "num-obj-points": 512,
    # "label-list-all" : ["lifting-platform", "minigolf-platform", "hammering-platform",
    #               "hammer", "spatula", "L-ruler", "U-tool", "fork-spatula", "asymmetric-L-ruler",
    #               "pipe-hammer", "pushing-platform"], # "reaching-platform"
    "label-list-all" : ["lifting-platform", "pushing-platform", "hammering-platform",
                "hammer", "spatula", "L-ruler"], # "reaching-platform"
    # "model-name" :  "waytu_unified_model_hammering_v25_best.pth",
    "model-name" : "waytu_unified_model_minigolf_v22_best.pth",
    "feature-extractor-path" : "small-pointner-encoder-distractor_best.pth",
    "feature-size" : 128,
    "num-trials" : 5, 
    "tool-type" : "primitive",
    # "tool-type" : "additional",
    # "task" : "hammering", 
    # "task" : "minigolf",
    "task" : "pushing",
    # "task" : "reaching",
    "dataset-save-path" : "test-dataset",
    "area-middle": {
        "min" : [-0.35 , 0.15, 0.060],
        "max" : [ 0.35 , 0.45, 0.065]
        },
    "area-negative" : {
        "min": [-0.50 , 0.15, 0.050],
        "max": [ -0.40 , 0.25, 0.060],
        },
    "area-positive" : {
        "min": [0.40 , 0.15, 0.050],
        "max": [ 0.50 , 0.25, 0.060],
        },
    "cameras" : ["camera1", "camera2", "camera3" ],
    "zero-shot-target": False

}

test_df = unified_model_selection_test(parameters)
print(test_df) 
